# Phase 5: scaled KB-on/off ablation (~30 images, Colab)

Runs the full loop over N smoke-subset images with their **COCO captions as
context prompts**: Qwen selection → KB colours (kb arm) + direct-colour
queries (llm arm) → Grounded-SAM masks → mask-measured re-resolution →
export for local three-arm scoring with `scripts/ablate_score.py`.

GPU: A100/L4 for the 7B model. ~30 images ≈ 60-90 min total.

> Runtime → GPU. Verify the clone is at your latest commit (printed below).

In [ ]:
!nvidia-smi -L
%cd /content
!test -d chroma-reasoner || git clone https://github.com/tomqi6195/chroma-reasoner.git
!cd chroma-reasoner && git pull && git log --oneline -1
!pip install -q -e chroma-reasoner
!pip install -q "transformers<5" accelerate pycocotools
import sys; sys.path.insert(0, '/content/chroma-reasoner/src')
import transformers; print('transformers', transformers.__version__)

## 1. Data: N smoke-subset images + captions as prompts

In [ ]:
import glob, json, os, urllib.request
import cv2

N_IMAGES = 30   # scale knob

# IMPORTANT: download the SAME deterministic 300-image subset as Phase 0 and
# slice the first N — sampling with n=30 directly would select different
# images than the 300-set on the local machine, breaking local scoring
# against the originals.
!cd chroma-reasoner && python scripts/download_coco_subset.py --n 300 --seed 0 --root /content/data
manifest = json.load(open('/content/data/manifest.json'))
IMAGE_IDS, CONTEXT = [], {}
for rec in manifest['images'][:N_IMAGES]:
    iid = rec['file_name'].split('.')[0]
    IMAGE_IDS.append(iid)
    CONTEXT[iid] = rec['captions'][0] if rec['captions'] else ''
print(len(IMAGE_IDS), 'images; e.g.', IMAGE_IDS[0], '->', CONTEXT[IMAGE_IDS[0]])

os.makedirs('gray', exist_ok=True)
for iid in IMAGE_IDS:
    img = cv2.imread(f'/content/data/val2017_subset/{iid}.jpg')
    cv2.imwrite(f'gray/{iid}.png', cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[:, :, 0])
print('grayscale ready')

## 2. Reason (kb arm) + direct colours (llm arm)

In [ ]:
from chroma_reasoner.kb import load_kb
from chroma_reasoner.reasoner import reason_plan
from chroma_reasoner.reasoner.backend_open import QwenVLBackend
from chroma_reasoner.reasoner.planner import ReasonerError
from chroma_reasoner.eval import llm_color_plan
from chroma_reasoner.plan import load_plan

def log(*parts):
    line = ' '.join(str(p) for p in parts)
    print(line)
    with open('phase5_log.txt', 'a') as lf:
        lf.write(line + '\n')

kb = load_kb('/content/chroma-reasoner/kb')
backend = QwenVLBackend(model_id='Qwen/Qwen2.5-VL-7B-Instruct')
print('backend ready')

In [ ]:
import traceback

os.makedirs('plans/reasoned', exist_ok=True)
os.makedirs('plans/ablation_llm', exist_ok=True)
ok = 0
for i, iid in enumerate(IMAGE_IDS):
    try:
        plan = reason_plan(kb, backend, f'gray/{iid}.png', user_prompt=CONTEXT[iid])
    except ReasonerError as e:
        log(f'!! {iid}: unrepairable:'); log(e); continue
    except Exception as e:
        log(f'!! {iid}: crashed: {type(e).__name__}: {e}'); continue
    with open(f'plans/reasoned/{iid}.json', 'w') as f:
        json.dump(plan, f, indent=2)
    try:
        ablated = llm_color_plan(plan, backend, f'gray/{iid}.png')
        with open(f'plans/ablation_llm/{iid}.json', 'w') as f:
            json.dump(ablated, f, indent=2)
    except Exception as e:
        log(f'!! {iid}: ablation failed: {type(e).__name__}: {e}')
    ok += 1
    log(f'[{i+1}/{len(IMAGE_IDS)}] {iid}: {len(plan["regions"])} regions  ({CONTEXT[iid][:50]!r})')
log(f'== reasoned {ok}/{len(IMAGE_IDS)} images')

In [ ]:
# Free the VLM before Grounded-SAM
import gc, torch
del backend
gc.collect(); torch.cuda.empty_cache()
print(f'free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 3. Ground + re-resolve

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import (AutoModelForZeroShotObjectDetection, AutoProcessor,
                          SamModel, SamProcessor)

device = 'cuda'
dino_proc = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-base')
dino = AutoModelForZeroShotObjectDetection.from_pretrained('IDEA-Research/grounding-dino-base').to(device).eval()
sam_proc = SamProcessor.from_pretrained('facebook/sam-vit-huge')
sam = SamModel.from_pretrained('facebook/sam-vit-huge').to(device).eval()

@torch.no_grad()
def phrase_to_mask(image_pil, phrase):
    text = phrase.lower().rstrip('.') + '.'
    inputs = dino_proc(images=image_pil, text=text, return_tensors='pt').to(device)
    out = dino(**inputs)
    res = dino_proc.post_process_grounded_object_detection(
        out, inputs.input_ids, threshold=0.25, text_threshold=0.2,
        target_sizes=[image_pil.size[::-1]])[0]
    if len(res['boxes']) == 0:
        return None, 0.0
    best = res['scores'].argmax()
    box = res['boxes'][best].tolist()
    s_in = sam_proc(image_pil, input_boxes=[[box]], return_tensors='pt').to(device)
    s_out = sam(**s_in)
    masks = sam_proc.image_processor.post_process_masks(
        s_out.pred_masks.cpu(), s_in['original_sizes'].cpu(), s_in['reshaped_input_sizes'].cpu())[0][0]
    scores = s_out.iou_scores.cpu()[0, 0]
    return masks[scores.argmax()].numpy().astype(bool), float(res['scores'][best])

In [ ]:
from chroma_reasoner.plan.masks import load_masks, region_key, save_mask
from chroma_reasoner.reasoner import re_resolve_with_masks

for plan_path in sorted(glob.glob('plans/reasoned/*.json')):
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    pil = Image.fromarray(np.stack([gray] * 3, axis=-1))
    for region in plan['regions']:
        mask, score = phrase_to_mask(pil, region['grounding_phrase'])
        if mask is None:
            log(f"!! {iid}/{region_key(region)}: NO DETECTION for {region['grounding_phrase']!r}"); continue
        save_mask(mask, 'masks_reasoned', iid, region)
    masks = load_masks('masks_reasoned', iid, plan, shape=gray.shape, allow_missing=True)
    if masks:
        plan = re_resolve_with_masks(kb, plan, gray, masks)
        with open(plan_path, 'w') as f:
            json.dump(plan, f, indent=2)
    log(f'{iid}: {len(masks)}/{len(plan["regions"])} masks')

## 4. Export

In [ ]:
!zip -rq phase5_outputs.zip plans/reasoned plans/ablation_llm masks_reasoned phase5_log.txt
from google.colab import files
files.download('phase5_outputs.zip')
# Locally (masks land in masks_reasoned/, subset originals already on disk):
#   python scripts/ablate_score.py --originals data/coco/val2017_subset \
#       --arm kb=plans/reasoned:masks_reasoned \
#       --arm llm=plans/ablation_llm:masks_reasoned \
#       --out results/phase5/scaled_scores.json